In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import pandas as pd

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

import PIL
from PIL import Image
import matplotlib.pyplot as plt

from torch.autograd import Variable


from torchvision.transforms import ToTensor
import torchvision.transforms as transforms
import torchvision.models as models


from sklearn import metrics, model_selection, preprocessing
from imblearn.over_sampling import SMOTE

ModuleNotFoundError: No module named 'sklearn.utils._metadata_requests'

In [2]:
torch.cuda.is_available()

False

In [3]:
dfx = pd.read_csv('../input/cassava-leaf-disease-classification/train.csv')

df_train, df_valid = model_selection.train_test_split(
        dfx, test_size=0.1, random_state=42, stratify=dfx.label.values
)


In [4]:
print(df_train.label.value_counts())

label
3    10370
4     2040
2     1882
1     1711
0      845
Name: count, dtype: int64


In [5]:
# X = df_train.drop('label', axis = 1)
# y = df_train['label']

# from sklearn.model_selection import train_test_split
# from sklearn.utils import resample

# train_data = pd.concat([X, y], axis=1)


# # separate minority and majority classes
# negative = train_data[train_data.label==3]

# x0 = train_data[train_data.label==0]
# x1 = train_data[train_data.label==1]
# x2 = train_data[train_data.label==2]
# x4 = train_data[train_data.label==4]

# # sampling = len(negative)
# sampling = 3000

# n0 = resample(x0, replace=True, n_samples=sampling, random_state=27) # reproducible results
# n1 = resample(x1, replace=True, n_samples=sampling, random_state=27) # reproducible results
# n2 = resample(x2, replace=True, n_samples=sampling, random_state=27) # reproducible results
# n4 = resample(x4, replace=True, n_samples=sampling, random_state=27) # reproducible results

# # combine minority and downsampled majority
# new_df_train = pd.concat([negative, n0, n1, n2, n4])
# print(new_df_train.label.value_counts())
# # check new class counts
# print(new_df_train)

In [6]:
# ## Downsampling ###
# X = df_train.drop('label', axis = 1)
# y = df_train['label']

# from sklearn.model_selection import train_test_split
# from sklearn.utils import resample

# train_data = pd.concat([X, y], axis=1)


# # separate minority and majority classes
# negative = train_data[train_data.label==3]

# x0 = train_data[train_data.label==0]
# x1 = train_data[train_data.label==1]
# x2 = train_data[train_data.label==2]
# x4 = train_data[train_data.label==4]

# total = len(x0) + len(x1) + len(x2) + len(x4)

# n0 = resample(negative, replace=True, n_samples=total, random_state=27) # reproducible results

# # combine minority and downsampled majority
# new_df_train = pd.concat([x0, x1, x2, x4, n0])
# print(new_df_train.label.value_counts())
# # check new class counts
# print(new_df_train)

In [7]:

_train = df_train.reset_index(drop=True)
df_valid = df_valid.reset_index(drop=True)

image_path = "../input/cassava-leaf-disease-classification/train_images/"
train_image_paths = [os.path.join(image_path, x) for x in df_train.image_id.values]
valid_image_paths = [os.path.join(image_path, x) for x in df_valid.image_id.values]

train_targets = df_train.label.values
valid_targets = df_valid.label.values




In [8]:
# """For Submission"""

dfx = pd.read_csv('../input/cassava-leaf-disease-classification/train.csv')

image_path = "../input/cassava-leaf-disease-classification/train_images/"
train_image_paths = [os.path.join(image_path, x) for x in dfx.image_id.values]
train_targets = dfx.label.values



In [9]:
len(train_image_paths),len(train_targets)

(18721, 18721)

In [10]:
len(valid_image_paths),len(valid_targets)

(1873, 1873)

In [11]:
# [7] - hyperparameter
"""torch module dataset"""
class CassavaDataset(Dataset):                    # Override torch.utils.data.Dataset
  def __init__(self, data, targets, transform=None):
    """
    Args:
      csv_file    (string): path of csv file
      dir         (string): path of images
      transoform  (callable, optional): Optional transform - IDK
    """
    self.files = data
    self.targets = targets
    self.classes = list(set(targets))
    self.transform = transform

  def __len__(self):
    return len(self.files)

  def __getitem__(self, idx):
    if torch.is_tensor(idx):
      idx = idx.tolist()
    name = self.files[idx]
    img_name = os.path.join(name)
    image = Image.open(img_name)
    """
    Need to transform image size
    ------------------------------------------------------------"""
    input_size = 384
    imagenet_stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
#     transform = transforms.Compose([transforms.Resize((input_size, input_size)),
#                                     transforms.Pad(8, padding_mode = 'reflect'),
#                                     transforms.RandomCrop(input_size),
#                                     transforms.ToTensor(),
#                                     transforms.Normalize(*imagenet_stats)])
    transform = transforms.Compose([transforms.RandomResizedCrop((input_size, input_size)),
                                    transforms.RandomHorizontalFlip(p=0.5),
                                    transforms.RandomVerticalFlip(p=0.5),
                                    transforms.ToTensor(),
                                    transforms.Normalize(*imagenet_stats)]
                                    )


    image = transform(image)

    """---------------------------------------------------------"""
#     label = self.cassava_info.iloc[idx, 1]
#     label = label.astype('int')
#     image = torch.Tensor(image).float()
#     sample = {'image': image, 'label': label}
    label = self.targets[idx]

    if self.transform:                    ## IDK
      sample = self.transform(sample)

    return image, label

In [12]:
# [8]
"""Dataset Initialization"""

cassava_data = CassavaDataset(train_image_paths, train_targets)

cassava_test = CassavaDataset(valid_image_paths, valid_targets)


In [13]:
# [9]
batch_size = 16

cassava_loader = DataLoader(cassava_data, batch_size=batch_size,
                            shuffle=True, num_workers=2)
classes = ('0', '1', '2', '3', '4')

test_loader = DataLoader(cassava_test, batch_size=batch_size,
                         shuffle=True, num_workers=2)

In [14]:
def denormalize(images, means, stds):
    if len(images.shape) == 3:
        images = images.unsqueeze(0)
    means = torch.tensor(means).reshape(1, 3, 1, 1)
    stds = torch.tensor(stds).reshape(1, 3, 1, 1)
    return images * stds + means

imagenet_stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
def show_image(img_tensor, label):
    print('Label:', cassava_data.classes[label], '(' + str(label) + ')')
    img_tensor = denormalize(img_tensor, *imagenet_stats)[0].permute((1, 2, 0))
    img_tensor = img_tensor[0].permute((1, 2, 0))
    plt.imshow(img_tensor)

def imshow(img, label):
#     img = img / 2 + 0.5
    npimg = img.numpy()
    print('Label:', cassava_data.classes[label], '(' + str(label) + ')')
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# RESNET


In [15]:
# # resnet = models.resnet34()
# resnet = models.resnext50_32x4d()


# num_ftrs = resnet.fc.in_features
# resnet.fc = nn.Linear(num_ftrs, 5)
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# resnet.to(device)

In [16]:
# [13]
"""optimizer setting"""
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(resnet.parameters(), lr=0.01, momentum=0.9)


NameError: name 'resnet' is not defined

In [17]:
# from tqdm import tqdm
# # [13]
# """ train """
# num_epochs = 1

# info_ground = [0, 0, 0, 0, 0]

# for epoch in range(num_epochs):
#     labels_size = 0
#     running_loss = 0.0
#     print('Epoch {}/{}'.format(epoch + 1, num_epochs))
    
#     resnet.train()
#     for batch in tqdm(cassava_loader):
        
#         images, labels = batch
#         images = images.to(device)
#         labels = labels.to(device)
        
#         out = resnet(images)                  # Generate predictions
#         loss = criterion(out, labels)  # Calculate loss   
        
#         labels_info = labels.squeeze()
#         labels_size += labels.size(0)

# #         total += labels.size(0)
    
#         for i in labels_info:
#             info_ground[i] += 1

        

#         loss.backward()
#         optimizer.step()
#         optimizer.zero_grad()
#         running_loss += loss.item()
    

#     epoch_loss = running_loss / labels_size
#     print('Loss: {:.4f}'.format(epoch_loss))
#     print('-' * 10)

# print('Finished Training')

In [18]:
# print(info_ground)

In [19]:
# # [14]
# """state dictionary save"""
# PATH = './cassava_net_resnext.pth'
# torch.save(resnet.state_dict(), PATH)

In [20]:
PATH = '../input/resnext/cassava_net_resnext.pth'
# PATH = './cassava_net_xtransform.pth'

# resnet = models.resnet34()
resnet =  models.resnext50_32x4d()

num_ftrs = resnet.fc.in_features
resnet.fc = nn.Linear(num_ftrs, 5)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet.to(device)

resnet.load_state_dict(torch.load(PATH))

FileNotFoundError: [Errno 2] No such file or directory: '../input/resnext/cassava_net_resnext.pth'

In [21]:
# [16]
"""Test code"""
correct = 0
total = 0

images_so_far = 0
num_images=20
fig = plt.figure()
resnet.eval()

info_ground = [0, 0, 0, 0, 0]
info_pred = [0, 0, 0, 0, 0]

with torch.no_grad():
    for batch in tqdm(test_loader):
        images, labels = batch
        images = images.to(device)
        labels = labels.to(device)
        out = resnet(images)                  # Generate predictions

        labels_info = labels.squeeze()
        
        _, predicted = torch.max(out.data, 1)
        
        predicted_info = predicted.squeeze()

        total += labels.size(0)
    
        for i in labels_info:
            info_ground[i] += 1
        for i in predicted_info:
            info_pred[i] += 1 

        correct += (predicted == labels).sum().item()


print('Accuracy of the network on the 2140 test images: %f %%' % (
    100 * correct/ total
))

NameError: name 'tqdm' is not defined

<Figure size 640x480 with 0 Axes>

In [22]:
print(info_ground)
print(info_pred)

[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]


In [23]:
# print(total)
# print(correct)

In [24]:
submission_df = pd.read_csv('../input/cassava-leaf-disease-classification/sample_submission.csv')
submission_df.head()

,image_id,label
0,1234294272.jpg,4
1,1234332763.jpg,4
2,1234375577.jpg,4
3,1234555380.jpg,4
4,1234571117.jpg,4


In [25]:
from PIL import Image

test_path = '/kaggle/input/cassava-leaf-disease-classification/test_images/'
test_images = os.listdir(test_path)
train_image_paths = [os.path.join(test_path, x) for x in test_images]

y_preds = []

for i in test_images:
    image = Image.open(f'/kaggle/input/cassava-leaf-disease-classification/test_images/{i}')
    input_size = 384
    transform = transforms.Compose([transforms.Resize((input_size, input_size)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(*imagenet_stats)]
                                    )
    
    image = transform(image)
    image = image.reshape(1, image.shape[0], image.shape[1], image.shape[2])
    image = Variable(image.to(device))
    out = resnet(image)                  # Generate predictions
    print(out)
    _, predicted = torch.max(out.data, 1)
    print(predicted.item())
    y_preds.append(predicted.item())

tensor([[ 0.0013, -0.0329,  0.0143,  0.0163,  0.0235]],
       grad_fn=<AddmmBackward0>)
4
tensor([[-0.0068, -0.0315,  0.0163,  0.0167,  0.0213]],
       grad_fn=<AddmmBackward0>)
4
tensor([[-0.0071, -0.0322,  0.0105,  0.0059,  0.0259]],
       grad_fn=<AddmmBackward0>)
4
tensor([[-0.0017, -0.0290,  0.0030,  0.0170,  0.0264]],
       grad_fn=<AddmmBackward0>)
4
tensor([[-0.0078, -0.0269,  0.0085, -0.0003,  0.0238]],
       grad_fn=<AddmmBackward0>)
4
tensor([[-0.0038, -0.0331,  0.0173,  0.0090,  0.0257]],
       grad_fn=<AddmmBackward0>)
4
tensor([[-0.0051, -0.0338,  0.0194,  0.0103,  0.0278]],
       grad_fn=<AddmmBackward0>)
4
tensor([[-0.0054, -0.0302,  0.0214,  0.0125,  0.0220]],
       grad_fn=<AddmmBackward0>)
4
tensor([[-0.0089, -0.0310,  0.0173,  0.0118,  0.0266]],
       grad_fn=<AddmmBackward0>)
4
tensor([[-0.0024, -0.0312,  0.0022,  0.0155,  0.0277]],
       grad_fn=<AddmmBackward0>)
4
tensor([[ 0.0007, -0.0336,  0.0044,  0.0216,  0.0297]],
       grad_fn=<AddmmBackward0>)
4

IsADirectoryError: [Errno 21] Is a directory: '/kaggle/input/cassava-leaf-disease-classification/test_images/test_images'

In [26]:
df_sub = pd.DataFrame({'image_id': test_images, 'label': y_preds})
display(df_sub)

ValueError: All arrays must be of the same length

In [27]:
df_sub.to_csv('submission.csv', index=None)

NameError: name 'df_sub' is not defined